# 00 — Datensatz-Partitionierung

Dieses Notebook erzeugt die drei disjunkten Datensatz-Splits, die für den gesamten Benchmark benötigt werden:

| Split | Zweck |
|---|---|
| **train** | NeuralFP-Training (Augmentation via MUSAN) |
| **ref** | Gemeinsame Referenz-Datenbank für alle drei Systeme |
| **ood_pool** | Out-of-Distribution-Queries (Song nicht in Ref-DB) |

Zusätzlich werden **Dry-Run-Subsets** (kleine Untermengen) für schnelle End-to-End-Tests angelegt.

**Abhängigkeiten:** Keine vorherigen Notebooks erforderlich.
**Ausgabe:** JSON-Dateien in `data/partitions/`


## 1. Imports und Pfade

Projekt-Root wird als übergeordnetes Verzeichnis von `notebooks/` bestimmt,
damit alle relativen Pfade unabhängig vom Start-Verzeichnis funktionieren.
`SEED = 42` gilt für alle Zufallsprozesse in diesem Notebook.


In [ ]:
import os, sys, json
from pathlib import Path

# Notebook lives in notebooks/; project root is one level up.
PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import (
    load_fma_metadata, create_partitions, create_dry_run_subsets,
    split_musan, filter_and_replenish_by_duration,
    assert_disjoint, print_genre_distribution, print_missing_files,
    save_partitions,
)

FMA_DIR        = PROJECT_ROOT / "data" / "fma_medium"
MUSAN_DIR      = PROJECT_ROOT / "data" / "musan"
PARTITIONS_DIR = PROJECT_ROOT / "data" / "partitions"
SEED = 42


## 2. FMA-Metadaten laden

Liest `fma_metadata/tracks.csv` (mehrstufiger Header), filtert auf das
`medium`-Subset und prüft, ob die .mp3-Datei jedes Tracks tatsächlich auf
dem Dateisystem vorhanden ist. Tracks ohne Audiodatei werden ausgeschlossen.

Ergebnis: DataFrame mit Spalten `genre`, `duration`, `filepath`, Index = `track_id`.


In [ ]:
track_df = load_fma_metadata(FMA_DIR)
print(f"Loaded: {len(track_df)} tracks")
print(track_df.head())


## 3. Dauer-Filter (≥ 30 s)

Tracks kürzer als 30 Sekunden werden ausgeschlossen, da sie zu kurz für
zuverlässige 10-Sekunden-Query-Segmente mit Puffer sind.


In [ ]:
before = len(track_df)
track_df = track_df[track_df["duration"] >= 30.0].copy()
print(f"Duration filter (>=30s): {before} → {len(track_df)} tracks "
      f"({before - len(track_df)} removed)")


## 4. Vollständige Partitionierung — Train / Ref / OOD-Pool

Genre-stratifiziertes Sampling in zwei Schritten:
1. `train` aus dem Gesamt-Pool ziehen
2. `ref` aus den verbleibenden Tracks ziehen
3. Rest → `ood_pool`

Da nur ~17 000 der 25 000 fma_medium-Tracks heruntergeladen sind,
werden die Zielgrößen proportional skaliert (~59 % / ~29 % / ~12 %).
`assert_disjoint` wird intern nach jedem Sampling aufgerufen.


In [ ]:
# fma_medium has ~25k entries but only ~17k audio files are downloaded.
# Scale partition sizes proportionally to what's available.
N_AVAIL = len(track_df)
N_TRAIN = cfg.n_train   # 15000
N_REF   = cfg.n_ref     # 8000
print(f"Available: {N_AVAIL}  →  n_train={N_TRAIN}, n_ref={N_REF}")

partitions = create_partitions(
    track_df, n_train=N_TRAIN, n_ref=N_REF, seed=SEED
)
print("Full partition sizes:", {k: len(v) for k, v in partitions.items()})


## 5. Dry-Run-Subsets ziehen (roh)

Kleine genre-proportionale Stichproben aus jedem der drei Splits:
- `dry_ref`:   ~50 Tracks  → Referenz-DB für schnellen Test
- `dry_train`: 200 Tracks  → NeuralFP-Training (2 Epochen)
- `dry_ood`:    10 Tracks  → OOD-Queries

Die Subsets werden direkt aus den Eltern-Splits entnommen und sind
per Konstruktion deren echte Teilmengen.


In [ ]:
dry_raw = create_dry_run_subsets(
    ref_ids      = partitions["ref"],
    train_ids    = partitions["train"],
    ood_pool_ids = partitions["ood_pool"],
    metadata_df  = track_df,
    n_ref=50, n_train=200, n_ood=10, seed=SEED,
)


## 6. Dauer-Prüfung der Dry-Run-Subsets und Auffüllen

Auch innerhalb der bereits gefilterten Partition können vereinzelt Tracks
unter 30 s liegen (z. B. wenn sich Metadaten-Dauer und tatsächliche Datei
unterscheiden). Diese werden durch zufällige Draws aus dem Rest der
jeweiligen Eltern-Partition ersetzt.


In [ ]:
# Build per-split pools (disjoint from each split)
ref_pool   = [t for t in partitions["ref"]      if t not in dry_raw["dry_ref"]]
train_pool = [t for t in partitions["train"]    if t not in dry_raw["dry_train"]]
ood_pool   = [t for t in partitions["ood_pool"] if t not in dry_raw["dry_ood"]]

dry_ref   = filter_and_replenish_by_duration(dry_raw["dry_ref"],   ref_pool,   track_df, 30.0, SEED)
dry_train = filter_and_replenish_by_duration(dry_raw["dry_train"], train_pool, track_df, 30.0, SEED+1)
dry_ood   = filter_and_replenish_by_duration(dry_raw["dry_ood"],   ood_pool,   track_df, 30.0, SEED+2)


## 7. MUSAN-Split (80 % Train / 20 % Eval)

MUSAN-Dateien aus den Kategorien `speech` und `noise` (nicht `music`)
werden deterministisch gemischt und aufgeteilt:
- **80 % train**: Noise-Augmentation beim NeuralFP-Training
- **20 % eval**:  Noise-Distortionen für Query-Generierung in NB 01

Die beiden Hälften sind strikt disjunkt — kein Data Leakage durch Noise.


In [ ]:
musan_split = split_musan(MUSAN_DIR, split=0.8, seed=SEED)
print(f"MUSAN train: {len(musan_split['train'])} files")
print(f"MUSAN eval:  {len(musan_split['eval'])}  files")


## 8. Data-Leakage-Check: Disjunktheit

Kritischste Validierung des gesamten Benchmarks. Drei Prüfungen:
1. Vollständige Splits (`train`, `ref`, `ood_pool`) sind paarweise disjunkt
2. Dry-Run-Subsets (`dry_ref`, `dry_train`, `dry_ood`) sind paarweise disjunkt
3. Jedes Dry-Subset ist echte Teilmenge seines Eltern-Splits

OOD-Songs dürfen **niemals** in der Referenz-DB oder im Training auftauchen —
andernfalls wäre die Specificity-Metrik wertlos.


In [ ]:
print("=== Full partitions ===")
assert_disjoint(partitions["train"], partitions["ref"], partitions["ood_pool"])

print("\n=== Dry subsets ===")
assert_disjoint(dry_ref, dry_train, dry_ood)

print("\n=== dry_ref ⊂ ref, dry_train ⊂ train, dry_ood ⊂ ood_pool ===")
assert set(dry_ref)   <= set(partitions["ref"]),      "dry_ref not subset of ref!"
assert set(dry_train) <= set(partitions["train"]),    "dry_train not subset of train!"
assert set(dry_ood)   <= set(partitions["ood_pool"]), "dry_ood not subset of ood_pool!"
print("Subset checks passed. ✓")


## 9. Genre-Verteilungen der Dry-Run-Subsets

Kontrolliert, ob die genre-stratifizierte Stichprobe die Verteilung des
Gesamt-Datensatzes (Rock ~35 %, Electronic ~30 %, ...) reproduziert.
Starke Abweichungen würden die Vergleichbarkeit der Systeme beeinträchtigen.


In [ ]:
print("--- dry_ref ---")
print_genre_distribution(dry_ref, track_df)
print("\n--- dry_train ---")
print_genre_distribution(dry_train, track_df)
print("\n--- dry_ood ---")
print_genre_distribution(dry_ood, track_df)


## 10. Fehlende Audio-Dateien prüfen

Stellt sicher, dass alle in die Dry-Run-Subsets aufgenommenen Tracks
tatsächlich als .mp3-Dateien auf dem Dateisystem vorliegen.
Fehlende Dateien würden in NB 01 zu stillen Fehlern führen.


In [ ]:
print("--- Missing files in dry_ref ---")
print_missing_files(dry_ref, track_df)
print("--- Missing files in dry_train ---")
print_missing_files(dry_train, track_df)
print("--- Missing files in dry_ood ---")
print_missing_files(dry_ood, track_df)


## 11. Partitionen persistieren

Alle Splits werden als JSON-Arrays in `data/partitions/` gespeichert.
Der MUSAN-Split wird separat als `musan_split.json` abgelegt
(enthält absolute Dateipfade statt Track-IDs).

Diese Dateien sind die einzige Datenquelle für alle nachfolgenden Notebooks —
**nie manuell bearbeiten**.


In [ ]:
all_partitions = {
    "train":    partitions["train"],
    "ref":      partitions["ref"],
    "ood_pool": partitions["ood_pool"],
    "dry_ref":   dry_ref,
    "dry_train": dry_train,
    "dry_ood":   dry_ood,
}
save_partitions(all_partitions, PARTITIONS_DIR)

# Save MUSAN split separately
musan_json = PARTITIONS_DIR / "musan_split.json"
with open(musan_json, "w") as fh:
    json.dump(musan_split, fh)
print(f"Saved musan_split.json → {musan_json}")


## 12. Abschließender Überblick (Sanity Check)

Zusammenfassung aller erzeugten Splits. Wird am Ende jedes Notebooks
ausgegeben, um bei späterem Nachvollziehen einen schnellen Überblick zu
ermöglichen.


In [ ]:
print("=== Sanity check ===")
print(f"train    : {len(partitions['train']):>6} tracks")
print(f"ref      : {len(partitions['ref']):>6} tracks")
print(f"ood_pool : {len(partitions['ood_pool']):>6} tracks")
print(f"dry_ref  : {len(dry_ref):>6} tracks")
print(f"dry_train: {len(dry_train):>6} tracks")
print(f"dry_ood  : {len(dry_ood):>6} tracks")

import pandas as pd
summary = pd.DataFrame({
    "set": ["train", "ref", "ood_pool", "dry_ref", "dry_train", "dry_ood"],
    "n":   [len(partitions["train"]), len(partitions["ref"]),
            len(partitions["ood_pool"]),
            len(dry_ref), len(dry_train), len(dry_ood)],
})
print(summary)
